In [ ]:
import os
import re
import cv2
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import shutil
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB3  
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import logging
logging.basicConfig(level=logging.INFO)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
dataset_path = "/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset"
train_benign_path = os.path.join(dataset_path, "train", "benign")
train_malignant_path = os.path.join(dataset_path, "train", "malignant")
test_benign_path = os.path.join(dataset_path, "test", "benign")
test_malignant_path = os.path.join(dataset_path, "test", "malignant")

train_path = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train'
validation_path = '/kaggle/working/validation'

In [ ]:
def count_images_in_folders(base_path, folder_list):
    counts = {}
    for folder in folder_list:
        folder_path = os.path.join(base_path, folder)
        if os.path.exists(folder_path):
            counts[folder] = len(os.listdir(folder_path)) 
        else:
            counts[folder] = 0
    return counts

folders = ["train/benign", "train/malignant", "test/benign", "test/malignant"]
image_counts = count_images_in_folders(dataset_path, folders)

for folder, count in image_counts.items():
    print(f"{folder}: {count} images")

In [ ]:
def check_filename_consistency(base_path, subfolders):
    for subfolder in subfolders:
        folder_path = os.path.join(base_path, subfolder)
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path): 
                if not re.match(r"melanoma_\d+\.jpg", filename):
                    print(f"Issue in {subfolder}: {filename}")

check_filename_consistency(dataset_path, folders)

In [ ]:
image_sizes = []
train_path = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train'
for class_name in os.listdir(train_path):
    class_path = os.path.join(train_path, class_name)
    for img_name in os.listdir(class_path)[:100]:  # Analyze first 100 images
        img = cv2.imread(os.path.join(class_path, img_name))
        image_sizes.append(img.shape[:2])  # Height, Width

image_sizes = np.array(image_sizes)
plt.scatter(image_sizes[:,1], image_sizes[:,0], alpha=0.5)
plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Image Size Distribution")
plt.show()


In [ ]:
def compute_mean_std(directory):
    images = []
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        for img_name in os.listdir(class_path)[:100]:
            img = load_img(os.path.join(class_path, img_name))
            img_array = img_to_array(img) / 255.0
            images.append(img_array)
    images = np.array(images)
    mean = np.mean(images, axis=(0, 1, 2))
    std = np.std(images, axis=(0, 1, 2))
    return mean, std

mean, std = compute_mean_std(train_path)
print(f"Mean: {mean}, Standard Deviation: {std}")

In [ ]:
os.makedirs(validation_path, exist_ok=True)  

def create_validation_split(train_path, validation_base_path, categories, split_ratio=0.2):
    for category in categories:
        source_folder = os.path.join(train_path, category)
        if not os.path.exists(source_folder):
            print(f"Source folder {source_folder} does not exist. Skipping...")
            continue

        target_folder = os.path.join(validation_base_path, category)
        os.makedirs(target_folder, exist_ok=True)

        images = os.listdir(source_folder)
        train_images, val_images = train_test_split(images, test_size=split_ratio, random_state=42)

        for val_image in val_images:
            src = os.path.join(source_folder, val_image)
            dst = os.path.join(target_folder, val_image)
            if not os.path.exists(dst):  # Avoid duplicates
                shutil.copy2(src, dst)

    print("Validation split created successfully!")
train_path = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train'
create_validation_split(train_path, validation_path, ["benign", "malignant"])


In [ ]:
train_files = set()
val_files = set()

for category in ["benign", "malignant"]:
    train_files.update(os.listdir(os.path.join(train_path, category)))
    val_files.update(os.listdir(os.path.join(validation_path, category)))


common_files = train_files.intersection(val_files)

print("Remaining duplicate images:", list(common_files))


for file in common_files:
    for category in ["benign", "malignant"]:
        val_file_path = os.path.join(validation_path, category, file)
        if os.path.exists(val_file_path):
            os.remove(val_file_path)
            print(f"🗑️ Deleted {file} from validation set!")



train_files = set()
val_files = set()

for category in ["benign", "malignant"]:
    train_files.update(os.listdir(os.path.join(train_path, category)))
    val_files.update(os.listdir(os.path.join(validation_path, category)))

common_files = train_files.intersection(val_files)

if len(common_files) == 0:
    print("No duplicate images left! Training and validation sets are fully cleaned.")
else:
    print(f"{len(common_files)} duplicate images still exist. Manual check needed!")


In [ ]:
train_benign = len(os.listdir(train_benign_path))
train_malignant = len(os.listdir(train_malignant_path))
test_benign = len(os.listdir(test_benign_path))
test_malignant = len(os.listdir(test_malignant_path))

print(f"train_benign: {train_benign} images")
print(f"train_malignant: {train_malignant} images")
print(f"test_benign: {test_benign} images")
print(f"test_malignant: {test_malignant} images")

In [ ]:
class_counts = {
    "Train Benign": train_benign,
    "Train Malignant": train_malignant,
    "Test Benign": test_benign,
    "Test Malignant": test_malignant
}


plt.figure(figsize=(8, 6))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()), palette='coolwarm')
plt.xlabel('Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.title('Class Distribution in Dataset', fontsize=14)
plt.xticks(rotation=45)
plt.show()

train_ratio = train_benign / train_malignant
test_ratio = test_benign / test_malignant

print(f"Train Benign to Malignant Ratio: {train_ratio:.2f}")
print(f"Test Benign to Malignant Ratio: {test_ratio:.2f}")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Augmentation for training data
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Only rescaling for validation data
valid_datagen = ImageDataGenerator(rescale=1./255)

# Path for training data — make sure these folders exist
train_dir = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/train'
val_dir = '/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset/test'

# Flow training images in batches
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True  # Shuffle is important for training
)

# Flow validation images in batches
val_generator = valid_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # Keep order for evaluation
)

# Check batch shape
x_train_batch, y_train_batch = next(train_generator)
x_val_batch, y_val_batch = next(val_generator)

print(f"Train batch shape: {x_train_batch.shape}, Labels shape: {y_train_batch.shape}")
print(f"Validation batch shape: {x_val_batch.shape}, Labels shape: {y_val_batch.shape}")


In [ ]:
def plot_images(generator, title, num_images=6):
    x_batch, y_batch = next(generator)
    
    plt.figure(figsize=(12, 6))
    plt.suptitle(title, fontsize=16)
    
    for i in range(num_images):
        plt.subplot(2, 3, i + 1)
        plt.imshow(x_batch[i])
        plt.axis('off')
    plt.show()

plot_images(train_generator, "Training Images (Should Be Augmented)")
plot_images(val_generator, "Validation Images (Should NOT Be Augmented)")

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255, 
    featurewise_center=True, 
    featurewise_std_normalization=True
)

datagen.mean = mean
datagen.std = std

train_dir = "/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/"

def show_augmented_images(generator, num_images=5):
    images, labels = next(generator)  
    plt.figure(figsize=(12, 6))
    for i in range(num_images):
        plt.subplot(1, num_images, i + 1)
        plt.imshow(images[i])  
        plt.axis("off")  
    plt.show()

show_augmented_images(train_generator)


In [ ]:
def plot_sample_images(dataset_dir, dataset_type, class_label, num_images=9):
    class_dir = os.path.join(dataset_dir, class_label)
    if not os.path.exists(class_dir):
        print(f"Directory not found: {class_dir}")
        return
    image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if not image_files:
        print(f" No images found in {class_dir}")
        return

    random.shuffle(image_files) 
    print(f" Displaying {min(num_images, len(image_files))} images from {dataset_type}/{class_label}")

    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    fig.suptitle(f"Sample Images - {dataset_type} ({class_label})", fontsize=14)

    for i, ax in enumerate(axes.flat):
        if i >= len(image_files):
            break

        img_path = os.path.join(class_dir, image_files[i])
        img = cv2.imread(img_path)

        if img is None:
            print(f"Could not load image: {img_path}")
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  
        ax.imshow(img)
        ax.axis("off")

    plt.show()

train_test_dir = "/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset"
validation_dir = "/kaggle/working/validation"

for dataset_type in ["train", "test"]:
    for class_label in ["benign", "malignant"]:
        plot_sample_images(os.path.join(train_test_dir, dataset_type), dataset_type, class_label, num_images=9)
for class_label in ["benign", "malignant"]:
    plot_sample_images(os.path.join(validation_dir), "validation", class_label, num_images=9)


In [ ]:
def normalize_dataset(dataset_path, target_size=(224, 224)):
   
    normalized_images = {}

    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):  
                image_path = os.path.join(root, file)
                image = load_img(image_path, target_size=target_size)  
                image_array = img_to_array(image) / 255.0  
                normalized_images[file] = image_array  
    return normalized_images

normalized_data = normalize_dataset(dataset_path)

print(f"Total images normalized: {len(normalized_data)}")

In [ ]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50, EfficientNetB0

# CNN Model
cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    Dense(1, activation='sigmoid')  
])

cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# ResNet50 Model (Transfer Learning)
base_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all layers initially
for layer in base_resnet.layers:
    layer.trainable = False

# Custom Fully Connected Layers
x = GlobalAveragePooling2D()(base_resnet.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

resnet_model = Model(inputs=base_resnet.input, outputs=output)

resnet_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# EfficientNetB0 Model (Transfer Learning)
base_efficientnet = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all layers initially
for layer in base_efficientnet.layers:
    layer.trainable = False

# Custom Fully Connected Layers
x = GlobalAveragePooling2D()(base_efficientnet.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

efficientnet_model = Model(inputs=base_efficientnet.input, outputs=output)

efficientnet_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Print Model Summaries
print("CNN Model Summary:")
cnn_model.summary()
print("\nResNet50 Model Summary:")
resnet_model.summary()
print("\nEfficientNetB0 Model Summary:")
efficientnet_model.summary()


In [ ]:
optimizer = Adam(learning_rate=0.0001)  

cnn_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
resnet_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
efficientnet_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# === PATH SETUP === #
dataset_path = "/kaggle/input/melanoma-skin-cancer-dataset-of-10000-images/melanoma_cancer_dataset"
train_dir = os.path.join(dataset_path, "train")
val_dir = "/kaggle/working/validation"

train_benign_path = os.path.join(train_dir, "benign")
train_malignant_path = os.path.join(train_dir, "malignant")
val_benign_path = os.path.join(val_dir, "benign")
val_malignant_path = os.path.join(val_dir, "malignant")

# === CREATE VALIDATION SPLIT === #
os.makedirs(val_benign_path, exist_ok=True)
os.makedirs(val_malignant_path, exist_ok=True)

def create_validation_split(train_path, val_path, split_ratio=0.2):
    images = os.listdir(train_path)
    train_images, val_images = train_test_split(images, test_size=split_ratio, random_state=42)

    for img in val_images:
        src = os.path.join(train_path, img)
        dst = os.path.join(val_path, img)
        if not os.path.exists(dst):  # Avoid re-copying
            shutil.copy(src, dst)

create_validation_split(train_benign_path, val_benign_path)
create_validation_split(train_malignant_path, val_malignant_path)

print("✅ Validation set created successfully!")

# === IMAGE AUGMENTATION === #
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator(rescale=1.0/255)

# === DATA LOADERS === #
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir, 
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# === CLASS WEIGHTS (for imbalance) === #
class_weights = compute_class_weight('balanced', classes=np.unique(train_data.classes), y=train_data.classes)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}

# === EfficientNetB3 MODEL === #
base_model = EfficientNetB3(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:100]:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
output_layer = Dense(1, activation='sigmoid')(x)

efficientnet_model = keras.Model(inputs=base_model.input, outputs=output_layer)

efficientnet_model.compile(optimizer=Adam(learning_rate=0.0001),
                           loss='binary_crossentropy',
                           metrics=['accuracy'])

history_effnet = efficientnet_model.fit(
    train_data,
    epochs=10,
    class_weight=class_weights_dict,
    validation_data=val_data
)

efficientnet_model.save("/kaggle/working/efficientnet_model.h5")
print("✅ EfficientNet Model Saved Successfully!")

# === SIMPLE CNN MODEL === #
cnn_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  
])

cnn_model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

history_cnn = cnn_model.fit(
    train_data,
    epochs=10,
    class_weight=class_weights_dict,
    validation_data=val_data
)

cnn_model.save("/kaggle/working/cnn_model.h5")
print("✅ CNN Model Saved Successfully!")

# === VERIFY SAVED MODELS === #
print("📁 Saved model files:", os.listdir("/kaggle/working/"))

# === LOAD MODEL FOR INFERENCE === #
loaded_model = tf.keras.models.load_model("/kaggle/working/efficientnet_model.h5")
print("✅ EfficientNet Model Loaded Successfully!")


In [ ]:
y_true = val_data.classes 
y_pred_probs = cnn_model.predict(val_data)  
y_pred = (y_pred_probs > 0.4).astype(int)  

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f" Accuracy: {accuracy:.4f}")
print(f" Precision: {precision:.4f}")
print(f" Recall: {recall:.4f}")
print(f" F1-score: {f1:.4f}")
print("\n Classification Report:\n", classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy', marker='o')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs. Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# hyperparameter tuning on the final model that is a combination of all architectures
import keras_tuner as kt
from tensorflow.keras import layers, models

def model_builder(hp):
    model = models.Sequential()
    model.add(tf.keras.applications.EfficientNetB0(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
        pooling='avg'
    ))

    # Dense layer
    hp_units = hp.Int('units', min_value=64, max_value=256, step=64)
    model.add(Dense(units=hp_units, activation='relu'))

    # Dropout layer
    hp_dropout = hp.Float('dropout', min_value=0.2, max_value=0.6, step=0.1)
    model.add(Dropout(rate=hp_dropout))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(
        optimizer=Adam(learning_rate=hp_learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
tuner = kt.Hyperband(
    model_builder,
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='my_tuner_dir',
    project_name='melanoma_hp_tuning'
)

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

tuner.search(train_generator, epochs=10, validation_data=val_generator, callbacks=[stop_early])


In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""
Best hyperparameters:
- Units: {best_hps.get('units')}
- Dropout: {best_hps.get('dropout')}
- Learning Rate: {best_hps.get('learning_rate')}
""")


In [ ]:
model = tuner.hypermodel.build(best_hps)
history = model.fit(train_generator, validation_data=val_generator, epochs=10)
